# Lesson 01 — TensorFlow Basics

> **Goal:** Understand tensors, shapes, basic ops, and automatic differentiation.  
> **Why it matters:** Every image, every weight in our face detector, every gradient update — all tensors.

---

## 1.1 — What is a Tensor?

You already know vectors and matrices from your maths background. A tensor is just the generalization:

| Rank | Name | Example |
|------|------|---------|
| 0 | Scalar | `5.0` — a single pixel brightness |
| 1 | Vector | `[1, 2, 3]` — a row of pixels |
| 2 | Matrix | `[[...], [...]]` — a grayscale image |
| 3 | 3D Tensor | `(height, width, 3)` — one RGB image |
| 4 | 4D Tensor | `(batch, height, width, 3)` — a batch of images |

When we feed 32 face images (each 224×224 pixels, RGB) into our model, the tensor shape is **(32, 224, 224, 3)**.

In [ ]:
import tensorflow as tf
import numpy as np

print("TensorFlow version:", tf.__version__)

In [ ]:
# --- Rank 0: Scalar ---
scalar = tf.constant(5.0)
print("Scalar:", scalar)
print("Shape:", scalar.shape)   # ()
print("Rank:",  tf.rank(scalar).numpy())  # 0

In [ ]:
# --- Rank 1: Vector ---
vector = tf.constant([1.0, 2.0, 3.0])
print("Vector:", vector)
print("Shape:", vector.shape)   # (3,)

In [ ]:
# --- Rank 2: Matrix ---
matrix = tf.constant([[1, 2, 3],
                       [4, 5, 6]])
print("Matrix:\n", matrix)
print("Shape:", matrix.shape)   # (2, 3)

In [ ]:
# --- Simulating a single grayscale image (8x8) ---
# In real life this would be 224x224 or 640x480
fake_grayscale_image = tf.ones([8, 8, 1])  # height=8, width=8, channels=1
print("Grayscale image shape:", fake_grayscale_image.shape)

# --- Simulating a single RGB image ---
fake_rgb_image = tf.ones([8, 8, 3])  # channels=3 (R, G, B)
print("RGB image shape:", fake_rgb_image.shape)

# --- Simulating a BATCH of 4 RGB images ---
fake_batch = tf.ones([4, 8, 8, 3])  # 4 images
print("Batch shape:", fake_batch.shape)

---
## 1.2 — Data Types (dtypes)

Tensors have a `dtype` — the numeric type of each element.

| dtype | Use case |
|-------|----------|
| `float32` | Most model weights and activations |
| `float16` | Mixed precision training (faster on GPUs) |
| `int32` | Labels, indices |
| `bool` | Masks |

**Image pixels are usually `uint8` (0–255) when loaded, but we convert to `float32` (0.0–1.0) before feeding into a model.** This is called *normalization*.

In [ ]:
# Raw pixel value (how images are stored on disk)
pixel_raw = tf.constant(128, dtype=tf.uint8)
print("Raw pixel:", pixel_raw, "dtype:", pixel_raw.dtype)

# Cast to float32 and normalize to [0, 1]
pixel_norm = tf.cast(pixel_raw, tf.float32) / 255.0
print("Normalized pixel:", pixel_norm.numpy(), "dtype:", pixel_norm.dtype)

---
## 1.3 — Basic Operations

TensorFlow ops map directly to the maths you already know.

In [ ]:
a = tf.constant([[1.0, 2.0],
                 [3.0, 4.0]])
b = tf.constant([[5.0, 6.0],
                 [7.0, 8.0]])

print("a + b (element-wise):\n", a + b)
print("\na * b (element-wise):\n", a * b)
print("\nMatrix multiply a @ b:\n", a @ b)   # same as tf.matmul(a, b)
print("\nTranspose of a:\n", tf.transpose(a))

In [ ]:
# Reduction ops — these appear everywhere in loss functions
x = tf.constant([1.0, 2.0, 3.0, 4.0])

print("sum:",  tf.reduce_sum(x).numpy())    # 10.0
print("mean:", tf.reduce_mean(x).numpy())   # 2.5
print("max:",  tf.reduce_max(x).numpy())    # 4.0

---
## 1.4 — Automatic Differentiation (The Magic Behind Training)

This is where your maths background pays off immediately.

In neural network training, we need to compute:
$$\frac{\partial L}{\partial w}$$
...the gradient of the loss `L` with respect to every weight `w`. We then update:
$$w \leftarrow w - \alpha \cdot \frac{\partial L}{\partial w}$$

TensorFlow does this automatically using `tf.GradientTape`. It records operations as they happen, then replays them backwards (chain rule) to compute gradients.

In [ ]:
# Simple example: f(x) = x^2, df/dx = 2x
x = tf.Variable(3.0)  # tf.Variable = a trainable tensor

with tf.GradientTape() as tape:
    f = x ** 2         # forward pass: compute f(x)

df_dx = tape.gradient(f, x)  # backward pass: compute df/dx

print(f"f(3) = {f.numpy()}")            # 9.0
print(f"df/dx at x=3: {df_dx.numpy()}") # 6.0  (= 2 * 3)

In [ ]:
# Now let's simulate ONE gradient descent step — the core of all neural net training

# Imagine w is one weight in our face detector
# True answer: y = 2x + 1. We're trying to learn w and b.

w = tf.Variable(0.0)    # start with wrong weight
b = tf.Variable(0.0)    # start with wrong bias
learning_rate = 0.1

# One training sample: input x=3.0, true output y=7.0  (= 2*3 + 1)
x_train = tf.constant(3.0)
y_true  = tf.constant(7.0)

print("Before training:")
print(f"  w={w.numpy()}, b={b.numpy()}, prediction={w*x_train + b:.2f}, true={y_true.numpy()}")

# Run 20 gradient descent steps
for step in range(20):
    with tf.GradientTape() as tape:
        y_pred = w * x_train + b          # forward pass
        loss   = (y_pred - y_true) ** 2   # MSE loss
    
    dw, db = tape.gradient(loss, [w, b])  # backprop
    
    w.assign_sub(learning_rate * dw)      # w = w - lr * dw
    b.assign_sub(learning_rate * db)      # b = b - lr * db

print("\nAfter 20 steps:")
print(f"  w={w.numpy():.4f} (want 2.0)")
print(f"  b={b.numpy():.4f} (want 1.0)")
print(f"  prediction={w*x_train + b:.4f} (want 7.0)")

---
## 1.5 — Reshaping Tensors

Reshaping is very common when connecting different layers of a network (e.g., flattening an image before a Dense layer).

In [ ]:
# A 2x4 image (tiny example)
img = tf.constant([[1, 2, 3, 4],
                   [5, 6, 7, 8]], dtype=tf.float32)
print("Original shape:", img.shape)   # (2, 4)

# Flatten to a 1D vector (what a Dense layer expects)
flat = tf.reshape(img, [-1])           # -1 means 'infer this dimension'
print("Flattened:", flat.numpy())      # [1. 2. 3. 4. 5. 6. 7. 8.]
print("Flat shape:", flat.shape)       # (8,)

# Add a batch dimension (model expects batches, not single images)
batched = tf.expand_dims(img, axis=0)  # (1, 2, 4)
print("Batched shape:", batched.shape)

---
## ✅ Lesson 01 — Summary

| Concept | What you learned |
|---------|------------------|
| Tensor | n-dimensional array; images are 3D/4D tensors |
| dtype | Use `float32` for model inputs; normalize pixels to [0,1] |
| Basic ops | `+`, `*`, `@`, `reduce_sum`, `reduce_mean` |
| `tf.Variable` | A trainable tensor (weights, biases) |
| `GradientTape` | Records ops → computes gradients automatically |
| Gradient descent | `w = w - lr * dw` — the core update rule |
| `reshape` | Change tensor shape without changing data |

---

## 🚀 Next: Lesson 02 — Build your first neural network (MNIST)

We'll use `tf.keras` to build, compile, and train a digit classifier in ~30 lines.  
Then in Lesson 03 we move to CNNs and you'll start seeing how face detection connects.